## field 목록

In [1]:
import json
import time
from pathlib import Path
from urllib.request import Request, urlopen

# 노트북 위치가 어디든 .env가 있는 프로젝트 루트를 찾기
cwd = Path.cwd()
PROJECT_ROOT = next(
    path for path in [cwd, *cwd.parents]
    if (path / ".env").exists()
)

def read_env_value(name: str) -> str:
    for line in (PROJECT_ROOT / ".env").read_text(encoding="utf-8").splitlines():
        if line.startswith(f"{name}="):
            return line.split("=", 1)[1].strip()
    raise ValueError(f"{name} 값을 .env에서 찾지 못했습니다.")

AKS_API_KEY = read_env_value("AKS_API_KEY")
API_BASE_URL = "https://devin.aks.ac.kr:8080/api"

In [4]:
from urllib.error import URLError, HTTPError

PAGE_SIZE = 100

def fetch_list_page(page_no: int, page_size: int = PAGE_SIZE) -> dict:
    url = f"{API_BASE_URL}/articles?p={page_no}&ps={page_size}"
    request = Request(
        url,
        headers={"X-API-Key": AKS_API_KEY},
        method="GET",
    )

    for attempt in range(3):
        try:
            with urlopen(request, timeout=90) as response:
                return json.loads(response.read().decode("utf-8"))

        except (TimeoutError, URLError) as error:
            if attempt == 2:
                raise error

            wait_seconds = (attempt + 1) * 3
            print(f"응답이 늦어 {wait_seconds}초 뒤 다시 시도합니다.")
            time.sleep(wait_seconds)

first_page = fetch_list_page(1)

print("전체 항목 수:", first_page["totalCount"])
print("전체 페이지 수:", first_page["totalPage"])

전체 항목 수: 75835
전체 페이지 수: 759


#### field 목록 확인

In [5]:
unique_fields = set()

for page_no in range(1, first_page["totalPage"] + 1):
    page = first_page if page_no == 1 else fetch_list_page(page_no, PAGE_SIZE)

    for item in page["items"]:
        field = item.get("field", "").strip()
        if field:
            unique_fields.add(field)

    print(f"{page_no}/{first_page['totalPage']} 페이지 확인 완료")
    time.sleep(0.2)  # API에 너무 빠르게 요청하지 않도록 잠깐 쉬기

sorted_fields = sorted(unique_fields)

print("\n중복 제거된 field 개수:", len(sorted_fields))
for field in sorted_fields:
    print("-", field)

1/759 페이지 확인 완료
2/759 페이지 확인 완료
3/759 페이지 확인 완료
4/759 페이지 확인 완료
5/759 페이지 확인 완료
6/759 페이지 확인 완료
7/759 페이지 확인 완료
8/759 페이지 확인 완료
9/759 페이지 확인 완료
10/759 페이지 확인 완료
11/759 페이지 확인 완료
12/759 페이지 확인 완료
13/759 페이지 확인 완료
14/759 페이지 확인 완료
15/759 페이지 확인 완료
16/759 페이지 확인 완료
17/759 페이지 확인 완료
18/759 페이지 확인 완료
19/759 페이지 확인 완료
20/759 페이지 확인 완료
21/759 페이지 확인 완료
22/759 페이지 확인 완료
23/759 페이지 확인 완료
24/759 페이지 확인 완료
25/759 페이지 확인 완료
26/759 페이지 확인 완료
27/759 페이지 확인 완료
28/759 페이지 확인 완료
29/759 페이지 확인 완료
30/759 페이지 확인 완료
31/759 페이지 확인 완료
32/759 페이지 확인 완료
33/759 페이지 확인 완료
34/759 페이지 확인 완료
35/759 페이지 확인 완료
36/759 페이지 확인 완료
37/759 페이지 확인 완료
38/759 페이지 확인 완료
39/759 페이지 확인 완료
40/759 페이지 확인 완료
41/759 페이지 확인 완료
42/759 페이지 확인 완료
43/759 페이지 확인 완료
44/759 페이지 확인 완료
45/759 페이지 확인 완료
46/759 페이지 확인 완료
47/759 페이지 확인 완료
48/759 페이지 확인 완료
49/759 페이지 확인 완료
50/759 페이지 확인 완료
51/759 페이지 확인 완료
52/759 페이지 확인 완료
53/759 페이지 확인 완료
54/759 페이지 확인 완료
55/759 페이지 확인 완료
56/759 페이지 확인 완료
57/759 페이지 확인 완료
58/759 페이지 확인 완료
59/759 페이지 확인 완료
60/759

#### 저장
`저장 완료: C:\SKN_AI\SKN33-3rd-1Team\data\reference\aks_field_catalog.json`

In [6]:
from datetime import datetime

field_catalog = {
    "source": "한국민족문화대백과사전 OpenAPI",
    "collected_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "field_count": len(sorted_fields),
    "fields": sorted_fields,
}

output_path = PROJECT_ROOT / "data" / "reference" / "aks_field_catalog.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

output_path.write_text(
    json.dumps(field_catalog, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("저장 완료:", output_path)

저장 완료: C:\SKN_AI\SKN33-3rd-1Team\data\reference\aks_field_catalog.json
